# 📈 Snorkel Intro Tutorial: Data Augmentation

In this tutorial, we will walk through the process of using *transformation functions* (TFs) to perform data augmentation.
Like the labeling tutorial, our goal is to train a classifier to YouTube comments as `SPAM` or `HAM` (not spam).
In the [previous tutorial](https://github.com/snorkel-team/snorkel-tutorials/blob/master/spam/01_spam_tutorial.ipynb),
we demonstrated how to label training sets programmatically with Snorkel.
In this tutorial, we'll assume that step has already been done, and start with labeled training data,
which we'll aim to augment using transformation functions.


Data augmentation is a popular technique for increasing the size of labeled training sets by applying class-preserving transformations to create copies of labeled data points.
In the image domain, it is a crucial factor in almost every state-of-the-art result today and is quickly gaining
popularity in text-based applications.
Snorkel models the data augmentation process by applying user-defined *transformation functions* (TFs) in sequence.
You can learn more about data augmentation in
[this blog post about our NeurIPS 2017 work on automatically learned data augmentation](https://snorkel.org/blog/tanda/).

The tutorial is divided into four parts:
1. **Loading Data**: We load a [YouTube comments dataset](http://www.dt.fee.unicamp.br/~tiago//youtubespamcollection/).
2. **Writing Transformation Functions**: We write Transformation Functions (TFs) that can be applied to training data points to generate new training data points.
3. **Applying Transformation Functions to Augment Our Dataset**: We apply a sequence of TFs to each training data point, using a random policy, to generate an augmented training set.
4. **Training a Model**: We use the augmented training set to train an LSTM model for classifying new comments as `SPAM` or `HAM`.

This next cell takes care of some notebook-specific housekeeping.
You can ignore it.

In [41]:
import os
import random

import numpy as np

# Turn off TensorFlow logging messages
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# For reproducibility
seed = 0
os.environ["PYTHONHASHSEED"] = str(seed)
np.random.seed(0)
random.seed(0)

The Youtube set data wasn't opening, so I added a couple of spam messages below to test my code


In [42]:
data = [
    ("Win a free iPhone now!!!", "spam"),
    ("Congratulations, claim your prize", "spam"),
    ("Call me when you reach home", "ham"),
    ("Let's meet tomorrow", "ham"),
    ("Limited time offer!!!", "spam"),
    ("Are you coming to class?", "ham")
]

If you want to display all comment text untruncated, change `DISPLAY_ALL_TEXT` to `True` below.

In [43]:
import pandas as pd


DISPLAY_ALL_TEXT = True

pd.set_option("display.max_colwidth", 0 if DISPLAY_ALL_TEXT else 50)

This next cell makes sure a spaCy English model is downloaded.
If this is your first time downloading this model, restart the kernel after executing the next cell.

In [44]:
 # Download the spaCy english model
! python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 38.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 1. Loading Data

We load the Kaggle dataset and create Pandas DataFrame objects for the `train` and `test` sets.
The two main columns in the DataFrames are:
* **`text`**: Raw text content of the comment
* **`label`**: Whether the comment is `SPAM` (1) or `HAM` (0).

For more details, check out the [labeling tutorial](https://github.com/snorkel-team/snorkel-tutorials/blob/master/spam/01_spam_tutorial.ipynb).

### Defining Missing `utils` Functions

The `ModuleNotFoundError` indicates that the `utils.py` file, containing essential functions for this tutorial, is not found. To resolve this, I'm providing definitions for these functions directly in the notebook.

Note: The `data` variable defined in cell `oFQ0yYnEBY0o` will be used by `load_spam_dataset`. Given the small size of this dataset, it will be split minimally for demonstration purposes.

In [45]:
import pandas as pd
from sklearn.model_selection import train_test_split

def load_spam_dataset(load_train_labels=True):
    """
    Loads a dataset of spam/ham comments.
    This version uses the 'data' variable provided by the user.
    """
    # Ensure 'data' variable is accessible (defined in cell oFQ0yYnEBY0o)
    global data

    # Convert 'spam'/'ham' to 1/0 labels
    processed_data = [
        (text, 1 if label == "spam" else 0) for text, label in data
    ]

    df_all = pd.DataFrame(processed_data, columns=["text", "label"])

    # Split the small dataset into train and test for demonstration
    # In a real scenario, you would have a larger dataset and a more robust split.
    if len(df_all) >= 2:
        df_train, df_test = train_test_split(df_all, test_size=2, random_state=42, stratify=df_all['label'])
    else:
        df_train = df_all.copy()
        df_test = pd.DataFrame(columns=["text", "label"])

    return df_train.reset_index(drop=True), df_test.reset_index(drop=True)


In [46]:
import pandas as pd

def preview_tfs(df, tfs, n_examples=2, n_tf_per_example=2):
    """
    Previews the effect of transformation functions on a DataFrame.
    Requires `spacy` preprocessor to be defined and functional for TFs to work.
    """
    print("--- Original Examples ---")
    display(df.head(n_examples))

    print("\n--- Transformed Examples ---")
    for i, row in df.head(n_examples).iterrows():
        print(f"\nOriginal: {row['text']} (Label: {row['label']})")
        # Create a Series object that mimics a data point expected by TFs
        x_data_point = pd.Series({"text": row["text"], "label": row["label"]})

        example_tf_results = []
        for tf_func in tfs:
            # TFs are decorated with @transformation_function(pre=[spacy])
            # When called directly, the preprocessor will run if the TF is designed for it.
            transformed_x = tf_func(x_data_point.copy()) # Pass a copy to avoid modifying original
            if transformed_x is not None:
                example_tf_results.append(transformed_x["text"])

        # Take up to n_tf_per_example transformations
        for j, transformed_text in enumerate(example_tf_results[:n_tf_per_example]):
            print(f"  TF {j+1}: {transformed_text}")


In [47]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

MAX_SEQUENCE_LENGTH = 15 # Adjusted for potentially longer comments
VOCAB_SIZE = 5000 # Adjusted for potentially larger vocabulary

def featurize_df_tokens(df, max_sequence_length=MAX_SEQUENCE_LENGTH, vocab_size=VOCAB_SIZE):
    """
    Featurizes text data using Tokenizer and pad_sequences for Keras input.
    """
    # Initialize tokenizer
    tokenizer = Tokenizer(num_words=vocab_size, oov_token="<unk>")

    # Fit on all text data from the DataFrame
    tokenizer.fit_on_texts(df["text"])

    # Convert text to sequences of integers
    sequences = tokenizer.texts_to_sequences(df["text"])

    # Pad sequences to ensure uniform input length
    padded_sequences = pad_sequences(sequences, maxlen=max_sequence_length)

    return padded_sequences

def get_keras_lstm(num_buckets, embedding_dim=100):
    """
    Returns a simple Keras LSTM model for binary classification.
    num_buckets here refers to the vocabulary size + 1 (for 0-padding).
    """
    model = Sequential([
        Embedding(num_buckets, embedding_dim, input_length=MAX_SEQUENCE_LENGTH),
        LSTM(128), # LSTM layer with 128 units
        Dense(1, activation='sigmoid') # Output layer for binary classification
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [48]:
import pandas as pd

data = [
    ("Win a free iPhone now!!!", "spam"),
    ("Congratulations! You won a lottery", "spam"),
    ("Call me when you reach home", "ham"),
    ("Are we meeting today?", "ham"),
    ("Limited time offer, claim now", "spam"),
    ("See you in class", "ham")
]

df = pd.DataFrame(data, columns=["text", "label"])

In [49]:
def custom_augmentation(text):
    return text + " !!! Limited offer"

augmented = []

for text, label in zip(df["text"], df["label"]):
    if label == "spam":
        augmented.append((custom_augmentation(text), label))

aug_df = pd.DataFrame(augmented, columns=["text", "label"])

In [50]:
final_df = pd.concat([df, aug_df])

print("Original size:", len(df))
print("After augmentation:", len(final_df))

final_df.head()

Original size: 6
After augmentation: 9


,text,label
0,Win a free iPhone now!!!,spam
1,Congratulations! You won a lottery,spam
2,Call me when you reach home,ham
3,Are we meeting today?,ham
4,"Limited time offer, claim now",spam


### Next Steps

1.  **Run the new cells above.** This will define the missing functions.
2.  **Comment out or remove the original import statements** for `utils` in cells `34xgywKMA_HI`, `OZhQq2yrA_HL`, and `TfV6SUIlA_HU`.
    *   For example, change `from utils import load_spam_dataset` to `# from utils import load_spam_dataset`.
3.  **Re-run the affected cells** (e.g., `34xgywKMA_HI`, `OZhQq2yrA_HL`, `TfV6SUIlA_HU`) and subsequent cells in the notebook.

### Addressing the `spaCy` Error

I noticed that the `spaCy` download in cell `zKSYUHPBA_HH` also encountered a traceback. If you continue to face issues with transformation functions (TFs) that use `spaCy` (like `change_person`, `swap_adjectives`, etc.), you might need to:

*   **Restart the Colab runtime** (Runtime -> Restart runtime).
*   **Rerun cell `zKSYUHPBA_HH`** to re-attempt the `spaCy` model download and installation.
*   Ensure that any `torch` related imports or installations in your environment are not conflicting, as the traceback indicated an issue there.

## 2. Writing Transformation Functions (TFs)

Transformation functions are functions that can be applied to a training data point to create another valid training data point of the same class.
For example, for image classification problems, it is common to rotate or crop images in the training data to create new training inputs.
Transformation functions should be atomic e.g. a small rotation of an image, or changing a single word in a sentence.
We then compose multiple transformation functions when applying them to training data points.

Common ways to augment text includes replacing words with their synonyms, or replacing names entities with other entities.
More info can be found
[here](https://towardsdatascience.com/data-augmentation-in-nlp-2801a34dfc28) or
[here](https://towardsdatascience.com/these-are-the-easiest-data-augmentation-techniques-in-natural-language-processing-you-can-think-of-88e393fd610).
Our basic modeling assumption is that applying these operations to a comment generally shouldn't change whether it is `SPAM` or not.

Transformation functions in Snorkel are created with the
[`transformation_function` decorator](https://snorkel.readthedocs.io/en/master/packages/_autosummary/augmentation/snorkel.augmentation.transformation_function.html#snorkel.augmentation.transformation_function),
which wraps a function that takes in a single data point and returns a transformed version of the data point.
If no transformation is possible, a TF can return `None` or the original data point.
If all the TFs applied to a data point return `None`, the data point won't be included in
the augmented dataset when we apply our TFs below.

Just like the `labeling_function` decorator, the `transformation_function` decorator
accepts `pre` argument for `Preprocessor` objects.
Here, we'll use a
[`SpacyPreprocessor`](https://snorkel.readthedocs.io/en/master/packages/_autosummary/preprocess/snorkel.preprocess.nlp.SpacyPreprocessor.html#snorkel.preprocess.nlp.SpacyPreprocessor).

In [51]:
!pip install snorkel spacy names

In [52]:
from snorkel.preprocess.nlp import SpacyPreprocessor

spacy = SpacyPreprocessor(text_field="text", doc_field="doc", memoize=True)

In [53]:
!pip install names

In [54]:
import names
from snorkel.augmentation import transformation_function

# Pregenerate some random person names to replace existing ones with
# for the transformation strategies below
replacement_names = [names.get_full_name() for _ in range(50)]


# Replace a random named entity with a different entity of the same type.
@transformation_function(pre=[spacy])
def change_person(x):
    person_names = [ent.text for ent in x.doc.ents if ent.label_ == "PERSON"]
    # If there is at least one person name, replace a random one. Else return None.
    if person_names:
        name_to_replace = np.random.choice(person_names)
        replacement_name = np.random.choice(replacement_names)
        x.text = x.text.replace(name_to_replace, replacement_name)
        return x


# Swap two adjectives at random.
@transformation_function(pre=[spacy])
def swap_adjectives(x):
    adjective_idxs = [i for i, token in enumerate(x.doc) if token.pos_ == "ADJ"] # pos: parts-of-speech
    # Check that there are at least two adjectives to swap.
    if len(adjective_idxs) >= 2:
        idx1, idx2 = sorted(np.random.choice(adjective_idxs, 2, replace=False))
        # Swap tokens in positions idx1 and idx2.
        x.text = " ".join(
            [
                x.doc[:idx1].text,
                x.doc[idx2].text,
                x.doc[1 + idx1 : idx2].text,
                x.doc[idx1].text,
                x.doc[1 + idx2 :].text,
            ]
        )
        return x

We add some transformation functions that use `wordnet` from [NLTK](https://www.nltk.org/) to replace different parts of speech with their synonyms.

In [55]:
import nltk
from nltk.corpus import wordnet as wn

nltk.download("wordnet")


def get_synonym(word, pos=None):
    """Get synonym for word given its part-of-speech (pos)."""
    synsets = wn.synsets(word, pos=pos)
    # Return None if wordnet has no synsets (synonym sets) for this word and pos.
    if synsets:
        words = [lemma.name() for lemma in synsets[0].lemmas()]
        if words[0].lower() != word.lower():  # Skip if synonym is same as word.
            # Multi word synonyms in wordnet use '_' as a separator e.g. reckon_with. Replace it with space.
            return words[0].replace("_", " ")


def replace_token(spacy_doc, idx, replacement):
    """Replace token in position idx with replacement."""
    return " ".join([spacy_doc[:idx].text, replacement, spacy_doc[1 + idx :].text])


@transformation_function(pre=[spacy])
def replace_verb_with_synonym(x):
    # Get indices of verb tokens in sentence.
    verb_idxs = [i for i, token in enumerate(x.doc) if token.pos_ == "VERB"]
    if verb_idxs:
        # Pick random verb idx to replace.
        idx = np.random.choice(verb_idxs)
        synonym = get_synonym(x.doc[idx].text, pos="v")
        # If there's a valid verb synonym, replace it. Otherwise, return None.
        if synonym:
            x.text = replace_token(x.doc, idx, synonym)
            return x


@transformation_function(pre=[spacy])
def replace_noun_with_synonym(x):
    # Get indices of noun tokens in sentence.
    noun_idxs = [i for i, token in enumerate(x.doc) if token.pos_ == "NOUN"]
    if noun_idxs:
        # Pick random noun idx to replace.
        idx = np.random.choice(noun_idxs)
        synonym = get_synonym(x.doc[idx].text, pos="n")
        # If there's a valid noun synonym, replace it. Otherwise, return None.
        if synonym:
            x.text = replace_token(x.doc, idx, synonym)
            return x


@transformation_function(pre=[spacy])
def replace_adjective_with_synonym(x):
    # Get indices of adjective tokens in sentence.
    adjective_idxs = [i for i, token in enumerate(x.doc) if token.pos_ == "ADJ"]
    if adjective_idxs:
        # Pick random adjective idx to replace.
        idx = np.random.choice(adjective_idxs)
        synonym = get_synonym(x.doc[idx].text, pos="a")
        # If there's a valid adjective synonym, replace it. Otherwise, return None.
        if synonym:
            x.text = replace_token(x.doc, idx, synonym)
            return x

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [56]:
tfs = [
    change_person,
    swap_adjectives,
    replace_verb_with_synonym,
    replace_noun_with_synonym,
    replace_adjective_with_synonym,
]

Let's check out a few examples of transformed data points to see what our TFs are doing.

We notice a couple of things about the TFs.

* Sometimes they make trivial changes (`"website"` to `"web site"` for replace_noun_with_synonym).
  This can still be helpful for training our model, because it teaches the model to be invariant to such small changes.
* Sometimes they introduce incorrect grammar to the sentence (e.g. `swap_adjectives` swapping `"young"` and `"more"` above).

The TFs are expected to be heuristic strategies that indeed preserve the class most of the time, but
[don't need to be perfect](https://arxiv.org/pdf/1901.11196.pdf).
This is especially true when using automated
[data augmentation techniques](https://snorkel.org/blog/tanda/)
which can learn to avoid particularly corrupted data points.
As we'll see below, Snorkel is compatible with such learned augmentation policies.

## 3. Applying Transformation Functions

We'll first define a `Policy` to determine what sequence of TFs to apply to each data point.
We'll start with a [`RandomPolicy`](https://snorkel.readthedocs.io/en/master/packages/_autosummary/augmentation/snorkel.augmentation.RandomPolicy.html)
that samples `sequence_length=2` TFs to apply uniformly at random per data point.
The `n_per_original` argument determines how many augmented data points to generate per original data point.

In [57]:
from snorkel.augmentation import RandomPolicy

random_policy = RandomPolicy(
    len(tfs), sequence_length=2, n_per_original=2, keep_original=True
)

In some cases, we can do better than uniform random sampling.
We might have domain knowledge that some TFs should be applied more frequently than others,
or have trained an [automated data augmentation model](https://snorkel.org/blog/tanda/)
that learned a sampling distribution for the TFs.
Snorkel supports this use case with a
[`MeanFieldPolicy`](https://snorkel.readthedocs.io/en/master/packages/_autosummary/augmentation/snorkel.augmentation.MeanFieldPolicy.html),
which allows you to specify a sampling distribution for the TFs.
We give higher probabilities to the `replace_[X]_with_synonym` TFs, since those provide more information to the model.

In [58]:
from snorkel.augmentation import MeanFieldPolicy

mean_field_policy = MeanFieldPolicy(
    len(tfs),
    sequence_length=2,
    n_per_original=2,
    keep_original=True,
    p=[0.05, 0.05, 0.3, 0.3, 0.3],
)

To apply one or more TFs that we've written to a collection of data points according to our policy, we use a
[`PandasTFApplier`](https://snorkel.readthedocs.io/en/master/packages/_autosummary/augmentation/snorkel.augmentation.PandasTFApplier.html)
because our data points are represented with a Pandas DataFrame.

In [59]:
from snorkel.augmentation import PandasTFApplier

tf_applier = PandasTFApplier(tfs, mean_field_policy)
#df_train_augmented = tf_applier.apply(df_train)
#Y_train_augmented = df_train_augmented["label"].values

In [60]:
#print(f"Original training set size: {len(df_train)}")
#print(f"Augmented training set size: {len(df_train_augmented)}")

We have almost doubled our dataset using TFs!
Note that despite `n_per_original` being set to 2, our dataset may not exactly triple in size,
because sometimes TFs return `None` instead of a new data point
(e.g. `change_person` when applied to a sentence with no persons).
If you prefer to have exact proportions for your dataset, you can have TFs that can't perform a
valid transformation return the original data point rather than `None` (as they do here).

## 4. Training A Model

Our final step is to use the augmented data to train a model. We train an LSTM (Long Short Term Memory) model, which is a very standard architecture for text processing tasks.

The next cell makes Keras results reproducible. You can ignore it.

In [61]:
import tensorflow as tf

# In TensorFlow 2.x, tf.keras.backend.set_session is deprecated/removed.
# For reproducibility in TensorFlow 2.x and Keras, use tf.random.set_seed.
tf.random.set_seed(0)

# The following lines are for TensorFlow 1.x session configuration and are not needed in TF2
# session_conf = tf.compat.v1.ConfigProto(
#     intra_op_parallelism_threads=1, inter_op_parallelism_threads=1
# )
# sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(), config=session_conf)
# tf.compat.v1.keras.backend.set_session(sess)

Now we'll train our LSTM on both the original and augmented datasets to compare performance.

So using the augmented dataset indeed improved our model!
There is a lot more you can do with data augmentation, so try a few ideas
out on your own!